# End-to-End LangChain RAG Baseline

| Field | Value |
|---|---|
| Stage | Foundations |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-24 |

Callout - Key idea:
A framework should standardize interfaces and composition without changing the experiment silently; wrapping the same retrieval math should preserve the same quality.

## 30-Second Summary

This notebook expresses the transparent TF-IDF baseline through LangChain's `Document`, `Embeddings`, `InMemoryVectorStore`, and `RunnableLambda` interfaces. It then verifies behavioral parity with the framework-free retriever and records retrieval, answer, citation, abstention, and latency signals.

## Why This Matters

Production code benefits from stable component interfaces, metadata-carrying documents, composable steps, and provider-swappable implementations. But adopting LangChain does not automatically improve relevance or grounding. We keep the corpus, vectors, questions, generator, and metrics constant so the framework's value and limitations remain clear.

## Scope

| Covers | Does not cover |
|---|---|
| LangChain documents, custom embeddings, in-memory vector store, two-step runnable, shared evaluation | Hosted LLMs, semantic embedding downloads, persistent indexes, tracing services |

### Prerequisites

Complete the mental-model and from-scratch notebooks. Run the project environment documented in `docs/SETUP.md`.

## Mental Model

```text
CorpusDocument -> LangChain Document -> custom Embeddings -> InMemoryVectorStore
                                                               |
question -> Runnable retrieval step -> ranked Documents -> answer step
                                                        -> answer + citation + abstention
```

LangChain supplies contracts and orchestration. Our code still owns the representation, retrieval configuration, answer policy, evaluation data, and acceptance thresholds. This separation lets later notebooks replace one component while keeping the rest of the experiment stable.

In [ ]:
from pathlib import Path
from time import perf_counter
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the repository.')


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.runnables import RunnableLambda
from langchain_core.vectorstores import InMemoryVectorStore

from rag_101 import (
    CorpusDocument, SearchResult, TfidfRetriever, TfidfVectorizer,
    evaluate_answers, evaluate_retrieval, extractive_answer,
    load_corpus, load_golden_questions,
)

documents = load_corpus()
questions = load_golden_questions()
len(documents), len(questions)

## How It Works

| Component | Contract | Our implementation | Later replacement |
|---|---|---|---|
| `Document` | Text, ID, metadata | Northstar policy record | Loader and chunker output |
| `Embeddings` | Text to vectors | Fitted TF-IDF vectors | Hosted/local semantic model |
| `VectorStore` | Add and similarity-search documents | In-memory cosine search | FAISS, Chroma, managed store |
| Runnable | Input-to-output step | Retrieve then extract/cite | Prompt, LLM, grader, router |
| Evaluation | Behavior against labels | Repository golden set | Larger reviewed regression suite |

The online architecture remains two-step RAG: retrieval always runs before answer construction. Its number of steps and failure boundaries are predictable.

## Baseline

The framework-free `TfidfRetriever` is the reference behavior. Before accepting the LangChain version, we capture its top result for every answerable golden question.

In [ ]:
python_retriever = TfidfRetriever(documents)
answerable_questions = [question for question in questions if question.relevant_doc_ids]
python_top_ids = {
    question.id: python_retriever.search(question.question, k=1)[0].document.id
    for question in answerable_questions
}
python_top_ids

## Technique Implementation

`TfidfEmbeddings` adapts the already-fitted educational vectorizer to LangChain's embedding interface. Fitting on the corpus before adding documents is essential: documents and queries must share one vocabulary and IDF space.

In [ ]:
class TfidfEmbeddings(Embeddings):
    def __init__(self, vectorizer: TfidfVectorizer):
        self.vectorizer = vectorizer

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self.vectorizer.transform(texts)

    def embed_query(self, text: str) -> list[float]:
        return self.vectorizer.transform_one(text)


vectorizer = TfidfVectorizer().fit(document.content for document in documents)
embedding = TfidfEmbeddings(vectorizer)
vector_store = InMemoryVectorStore(embedding=embedding)

langchain_documents = [
    Document(
        id=document.id,
        page_content=document.content,
        metadata={**document.metadata, 'title': document.title},
    )
    for document in documents
]
vector_store.add_documents(langchain_documents, ids=[document.id for document in documents])


def retrieve(question: str, k: int = 3) -> list[SearchResult]:
    pairs = vector_store.similarity_search_with_score(question, k=k)
    return [
        SearchResult(
            document=CorpusDocument(
                id=str(document.id),
                title=str(document.metadata['title']),
                content=document.page_content,
                metadata={key: value for key, value in document.metadata.items() if key != 'title'},
            ),
            score=score,
        )
        for document, score in pairs
    ]


retrieval_step = RunnableLambda(
    lambda question: {'question': question, 'results': retrieve(question, k=3)}
)
answer_step = RunnableLambda(
    lambda payload: {
        'question': payload['question'],
        'retrieved_ids': [item.document.id for item in payload['results']],
        **extractive_answer(payload['question'], payload['results']),
    }
)
rag_chain = retrieval_step | answer_step

rag_chain.invoke('How long are customer conversation transcripts retained by default?')

## Controlled Experiment

The framework-free and LangChain retrievers use identical documents, tokenization, vocabulary, IDF weights, cosine similarity, and questions. We compare the top document ID for every answerable question. Behavioral parity is success: framework composition should not be credited with a quality gain it did not create.

In [ ]:
langchain_top_ids = {
    question.id: retrieve(question.question, k=1)[0].document.id
    for question in answerable_questions
}
parity = {
    question_id: python_top_ids[question_id] == langchain_top_ids[question_id]
    for question_id in python_top_ids
}

start = perf_counter()
runs = [rag_chain.invoke(question.question) for question in questions]
elapsed_ms = (perf_counter() - start) * 1000

experiment_results = {
    'top_1_parity': sum(parity.values()) / len(parity),
    'questions': len(runs),
    'total_latency_ms': round(elapsed_ms, 3),
    'mean_latency_ms': round(elapsed_ms / len(runs), 3),
}
experiment_results

## Evaluation

This scorecard is intentionally small and deterministic. Retrieval is evaluated only on answerable questions; abstention is evaluated on the explicitly unsupported question. Token-F1 is a lexical overlap proxy, not a complete measure of answer quality.

| Signal | Acceptance threshold | Baseline result |
|---|---:|---:|
| Hit rate@3 | 1.00 | 1.00 |
| Recall@3 | 1.00 | 1.00 |
| MRR | 1.00 | 1.00 |
| Citation accuracy | 1.00 | 1.00 |
| Abstention accuracy | 1.00 | 1.00 |
| Mean token-F1 | >= 0.45 | approximately 0.78 |

Later lessons must rerun this scorecard and add relevant latency/cost measurements.

In [ ]:
retrieval_metrics = evaluate_retrieval(
    questions,
    lambda question, k: [item.document.id for item in retrieve(question, k)],
    k=3,
)
answer_metrics = evaluate_answers(questions, rag_chain.invoke)
scorecard = {**retrieval_metrics, **answer_metrics, **experiment_results}

assert scorecard['top_1_parity'] == 1.0
assert scorecard['hit_rate@3'] == 1.0
assert scorecard['mrr'] == 1.0
assert scorecard['citation_accuracy'] == 1.0
assert scorecard['abstention_accuracy'] == 1.0
assert scorecard['mean_token_f1'] >= 0.45
scorecard

## Decision Guide

| Situation | Choose | Reason | Trade-off |
|---|---|---|---|
| Learning or diagnosing retrieval math | Framework-free baseline | Full visibility | More assembly code |
| Swapping providers/stores and composing steps | LangChain interfaces | Standard contracts and reusable orchestration | More abstraction and version surface |
| Small deterministic tests | In-memory vector store | Fast and disposable | No durability or distributed scale |
| Production knowledge service | Persistent authorized store | Lifecycle, filtering, operations | Infrastructure and migration complexity |

A framework is justified when its interfaces reduce system complexity—not when it hides an unmeasured pipeline.

## Failure Modes and Debugging

| Symptom | Likely cause | How to verify | Fix |
|---|---|---|---|
| Framework result differs from reference | Different preprocessing, score semantics, or tie ordering | Compare vectors and top IDs stage by stage | Align one variable at a time |
| Query vector has wrong dimension | Vectorizer refitted or provider mismatch | Log index/query model and dimension | Use one fitted/versioned representation |
| Metadata disappears | Conversion omitted IDs or metadata | Inspect returned `Document` | Enforce a document metadata contract |
| Good demo, poor production result | Golden set too small/easy | Segment real failures and expand labels | Maintain reviewed regression data |
| Notebook breaks after upgrade | Import/API version drift | Run smoke tests in the locked environment | Pin, migrate, and document versions |

## Production Notes

### Observability
Trace runnable boundaries, retrieval latency, ranked IDs/scores, selected context, answer/citation, configuration versions, and evaluation labels.

### Safety and Guardrails
Replace the in-memory store with a system that enforces tenant and document permissions before ranking. Do not log sensitive content by default.

### Latency and Cost
This local baseline has no provider cost and is fast because the corpus is tiny. Hosted embeddings and LLMs add network latency, token cost, rate limits, and retry behavior that must be measured separately.

## Practice

Replace `TfidfEmbeddings` with one semantic embedding provider while keeping documents, golden questions, `k`, generator, and metrics unchanged. Predict which query improves, record latency, and explain whether the added dependency is justified.

## Recall

Toggle - Recall: What value did LangChain add here?
Standard document, embedding, vector-store, and runnable interfaces—not an automatic relevance improvement.

Toggle - Recall: Why fit the vectorizer before indexing?
Documents and queries must use the same vocabulary and IDF weights.

Toggle - Recall: Why require parity with the from-scratch baseline?
It catches silent behavior changes introduced while adapting to framework interfaces.

Toggle - Recall: What should change in the semantic-embedding experiment?
Only the representation/retrieval component; the dataset, questions, generator, and metrics stay fixed.

## Sources

- [LangChain retrieval documentation](https://docs.langchain.com/oss/python/deepagents/retrieval)
- [LangChain `InMemoryVectorStore` reference](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore)
- [LangChain `Embeddings` reference](https://reference.langchain.com/python/langchain-core/embeddings/Embeddings)
- [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-24 | Complete | High | Revalidate on the next locked LangChain upgrade |